# Sensitivitätsanalyse – Portfolio-Layer Grid Search

**Kein Neutraining.** Lädt die gespeicherten Checkpoints aus dem `trading-results` Dataset
und führt einen Grid Search über Portfolio-Parameter durch.

**Benötigte Datasets (Add data → Your datasets):**
- `busersteven/trading-results` — enthält `kaggle_artifacts.tar.gz` (Checkpoints + Metadaten)
- `busersteven/trading-raw-data` — enthält die Parquet-Kursdaten

**Accelerator:** CPU reicht (kein Training, nur Inferenz + Portfolio-Simulation)

**Laufzeit:** ca. 30–60 Min (Phase 1: Scores einmalig berechnen) + ca. 1 Min (Phase 2: 81 Grid-Kombinationen)

| Parameter | Werte |
|---|---|
| `n_max` | 5, 7, 9 |
| `rotation_buffer` | 2, 3, 4 |
| `hard_stop_pct` | 20%, 25%, 30% |
| `fees` | 0.1%, 0.15%, 0.2% |

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import tarfile
import time
from pathlib import Path

WORKING  = Path('/kaggle/working')
REPO_DIR = WORKING / 'repo'

# ── Repo klonen ───────────────────────────────────────────────────────────────
if not REPO_DIR.exists():
    r = subprocess.run(
        ['git', 'clone', '--depth=1',
         'https://github.com/stevenlangeshops/trading.git', str(REPO_DIR)],
        capture_output=True, text=True,
    )
    print(r.stdout or 'clone ok')
else:
    print('Repo bereits vorhanden')

sys.path.insert(0, str(REPO_DIR))
os.chdir(str(REPO_DIR))

# ── Dependencies ─────────────────────────────────────────────────────────────
subprocess.run(
    [sys.executable, '-m', 'pip', 'install',
     'ta==0.11.0', 'loguru==0.7.2', '--quiet', '--no-warn-script-location'],
    capture_output=True,
)
print('Dependencies ok')

# ── Artefakte lokalisieren ────────────────────────────────────────────────────
# Kaggle speichert Dataset-Dateien bereits entpackt – kein tar.gz nötig.
# Suche nach fold_*_best.pt direkt im Input-Verzeichnis.
EXTRACT_DIR = WORKING / 'artifacts'
EXTRACT_DIR.mkdir(exist_ok=True)

# Erst prüfen ob Checkpoints direkt im Input liegen (Dataset-Upload entpackt)
pt_in_input = list(Path('/kaggle/input').rglob('fold_0_best.pt'))
if pt_in_input:
    # Checkpoints liegen direkt im Dataset-Ordner → symlink/copy in EXTRACT_DIR
    src_dir = pt_in_input[0].parent
    for f in src_dir.iterdir():
        dst = EXTRACT_DIR / f.name
        if not dst.exists():
            shutil.copy(f, dst)
    print(f'Artefakte aus Dataset kopiert: {src_dir}')
else:
    # Fallback: tar.gz suchen und entpacken
    tar_path = next((p for p in Path('/kaggle/input').rglob('kaggle_artifacts.tar.gz')), None)
    assert tar_path, (
        'Weder fold_*_best.pt noch kaggle_artifacts.tar.gz gefunden.\n'
        'Bitte trading-results Dataset hinzufuegen (Add data → Your datasets).'
    )
    with tarfile.open(tar_path) as tf:
        tf.extractall(str(EXTRACT_DIR))
    print(f'Entpackt: {tar_path.name} ({tar_path.stat().st_size // 1024} KB)')

print('Artefakte:', sorted(p.name for p in EXTRACT_DIR.iterdir())[:20])

In [ ]:
# ── Parquet-Daten ins Repo kopieren ──────────────────────────────────────────
raw_dest = REPO_DIR / 'data' / 'raw'
raw_dest.mkdir(parents=True, exist_ok=True)

parquet_files = list(Path('/kaggle/input').rglob('*.parquet'))
assert parquet_files, 'Keine Parquet-Dateien gefunden – trading-raw-data Dataset hinzugefuegt?'

for f in parquet_files:
    shutil.copy(f, raw_dest / f.name)
print(f'{len(parquet_files)} Parquet-Dateien nach {raw_dest} kopiert')

# ── Checkpoint-Pfade auflösen ─────────────────────────────────────────────────
# Checkpoints liegen flat im EXTRACT_DIR (fold_0_best.pt ... fold_11_best.pt)
pt_files = list(EXTRACT_DIR.glob('fold_*_best.pt'))
if not pt_files:                                    # verschachtelt?
    pt_files = list(EXTRACT_DIR.rglob('fold_*_best.pt'))
ckpt_dir = pt_files[0].parent if pt_files else EXTRACT_DIR

wf_json       = next(EXTRACT_DIR.rglob('v2_7d_walk_forward.json'))
asset_map_path = next(EXTRACT_DIR.rglob('asset_map.json'))

print(f'Checkpoints : {ckpt_dir}  ({len(pt_files)} .pt Dateien)')
print(f'walk_forward: {wf_json}')
print(f'asset_map   : {asset_map_path}')

In [ ]:
# ── Phase 1: Score-Cache aufbauen (einmalig) ──────────────────────────────────
# Lädt jeden Fold-Checkpoint einmal, berechnet Scores fuer alle Out-of-Sample-Tage.
# Danach wird das Modell nicht mehr benoetigt.

from run_sensitivity import (
    load_walk_forward_json, load_asset_map,
    build_features_from_parquet, build_price_cache_local,
    build_score_cache,
)

fold_results = load_walk_forward_json(str(wf_json))
for fold in fold_results:
    fold['ckpt_path'] = str(ckpt_dir / Path(fold['ckpt_path']).name)

asset_map   = load_asset_map(str(asset_map_path))
features    = build_features_from_parquet(str(raw_dest))
price_cache = build_price_cache_local(asset_map, raw_dest)

print('\nPhase 1: Score-Cache berechnen (dauert auf CPU ~30-60 Min) ...')
score_cache = build_score_cache(features, fold_results, asset_map)
print(f'Score-Cache fertig: {len(score_cache)} Handelstage')

In [ ]:
# ── Phase 2: Grid Search (81 Kombinationen) + Tearsheet ──────────────────────
from run_sensitivity import (
    grid_search, plot_top_equity_curves, print_summary,
    PARAM_GRID, compute_daily_ic, rolling_ic_report, subperiod_report,
)

df = grid_search(score_cache, price_cache, PARAM_GRID)

df.to_csv(str(WORKING / 'sensitivity_results.csv'))
plot_top_equity_curves(
    score_cache, price_cache, df,
    save_path=str(WORKING / 'sensitivity_top_equity.png'),
)

# Vollständiges Tearsheet: Top-Tabelle + Subperioden + Rolling IC
print_summary(df, score_cache=score_cache, price_cache=price_cache, horizon=7)

In [ ]:
# ── Ergebnisse anzeigen ───────────────────────────────────────────────────────
import pandas as pd
from IPython.display import Image, display

results = pd.read_csv(str(WORKING / 'sensitivity_results.csv'), index_col=0)
display(
    results.head(20).style
    .background_gradient(subset=['sharpe'],          cmap='Greens')
    .background_gradient(subset=['total_return_%'],  cmap='Blues')
    .background_gradient(subset=['max_drawdown_%'],  cmap='Reds_r')
    .format({
        'hard_stop_pct':  '{:.0%}',
        'fees':           '{:.3%}',
        'sharpe':         '{:.3f}',
        'total_return_%': '{:+.1f}%',
        'max_drawdown_%': '{:.1f}%',
        'win_rate_%':     '{:.1f}%',
    })
)
display(Image(str(WORKING / 'sensitivity_top_equity.png')))

In [ ]:
# ── Ergebnisse im Dataset speichern (optional) ────────────────────────────────
kaggle_key = None
try:
    from kaggle_secrets import UserSecretsClient
    kaggle_key = UserSecretsClient().get_secret('KAGGLE_KEY')
except Exception:
    pass

if kaggle_key:
    cfg = Path('/root/.kaggle/kaggle.json')
    cfg.parent.mkdir(parents=True, exist_ok=True)
    cfg.write_text(json.dumps({'username': 'busersteven', 'key': kaggle_key}))
    cfg.chmod(0o600)

    up = WORKING / 'sensitivity_upload'
    up.mkdir(exist_ok=True)
    for fname in ['sensitivity_results.csv', 'sensitivity_top_equity.png']:
        src = WORKING / fname
        if src.exists():
            shutil.copy(src, up / fname)

    (up / 'dataset-metadata.json').write_text(json.dumps({
        'title': 'trading-results', 'id': 'busersteven/trading-results',
        'licenses': [{'name': 'other'}],
    }))
    r = subprocess.run(
        ['kaggle', 'datasets', 'version', '-p', str(up),
         '-m', f'Sensitivity {time.strftime("%Y%m%d_%H%M%S")}', '--dir-mode', 'zip'],
        capture_output=True, text=True,
    )
    print(r.stdout or r.stderr)
else:
    print('Kein KAGGLE_KEY – Dateien manuell herunterladen:')
    print('  /kaggle/working/sensitivity_results.csv')
    print('  /kaggle/working/sensitivity_top_equity.png')